# C 언어 문제은행 — 2026년 **2회** 유형

> 2026년 2회 실기에서 새로 드러난 함정만 모았습니다.
> 이 회차의 C 언어 문제는 구조체 트리 **후위 순회**, 재귀의 **음수 인자**, 값/주소/배열 전달의 차이 를 물었습니다.
>
> 1회가 "언어가 **언제 무엇을 결정하는가**"였다면,
> 2회는 **"끝까지 손으로 따라갈 수 있는가"**를 물었습니다.
>
> **규칙**
> 1. 정답 토글을 열기 전에 **반드시 종이에 적는다.**
> 2. 재귀와 트리는 **호출 트리를 그려서** 아래에서 위로 값을 채운다. 암산하면 반드시 틀린다.
> 3. 먼저 `2026_1회_유형` 폴더를 끝낸 뒤에 이쪽으로 온다.

---

## Q1. 구조체 트리 후위 순회 (26년2회 13번 유형)

```c
#include <stdio.h>

typedef struct N {
    int v;
    struct N *a;
    struct N *b;
} N;

int cnt = 0;

void visit(N *n) {
    if (n == NULL) return;
    visit(n->a);
    visit(n->b);
    cnt++;
    printf("%d ", n->v);
}

int main(void) {
    N d = {7, NULL, NULL};
    N e = {9, NULL, NULL};
    N c = {53, NULL, NULL};
    N b = {12, &d, &e};
    N a = {64, &b, &c};
    visit(&a);
    printf("| %d", cnt);
    return 0;
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
7 9 12 53 64 | 5
```

**후위 순회(post-order)** 다 — 왼쪽 → 오른쪽 → **자기 자신** 순으로 방문한다.

트리 모양:
```
        a(64)
       /     \
    b(12)   c(53)
    /   \
  d(7)  e(9)
```

방문 순서를 손으로 따라가면 : d(7) → e(9) → b(12) → c(53) → a(64)
`cnt` 는 방문할 때마다 증가하므로 노드 수 **5**.

> 순회 종류를 구분할 것. **전위**는 자기 → 왼 → 오, **중위**는 왼 → 자기 → 오, **후위**는 왼 → 오 → 자기.
> `printf` 가 재귀 호출 **앞**에 있으면 전위, **뒤**에 있으면 후위다. 위치만 보면 바로 판별된다.

</details>

## Q2. 재귀 — 음수 인자가 들어가는 경우 (26년2회 4번 유형)

```c
#include <stdio.h>

int f(int n) {
    if (n <= 1) return n;
    return f(n - 3) + f(n - 1);
}

int main(void) {
    printf("%d %d", f(5), f(6));
    return 0;
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
1 1
```

함정은 `n - 3` 때문에 **인자가 음수까지 내려간다**는 것이다.
종료 조건이 `n <= 1` 이라 음수도 걸리는데, 그때 **0이 아니라 n 자신을 반환**한다.

f(5) = f(2) + f(4)
- f(2) = f(-1) + f(1) = **-1** + 1 = 0
- f(4) = f(1) + f(3) = 1 + (f(0) + f(2)) = 1 + (0 + 0) = 1
- f(5) = 0 + 1 = **1**

f(6) = f(3) + f(5) = 0 + 1 = **1**

`f(-1)` 을 0으로 착각하면 전부 틀린다. **종료 조건이 무엇을 반환하는지**를 먼저 확인할 것.

</details>

## Q3. 값 전달 · 주소 전달 · 배열 전달 (26년2회 7번 유형)

```c
#include <stdio.h>

void f1(int *p) { *p = 50; }
void f2(int p)  { p = 60; }
void f3(int p[]) { p[0] = 99; }

int main(void) {
    int i = 30;
    int arr[] = {2, 4, 6, 8, 10};
    int *q = arr;

    f1(&i);  printf("%d ", i);
    f2(i);   printf("%d ", i);
    f3(arr); printf("%d ", arr[0]);
    printf("%d %d", *q, *(q + 3));
    return 0;
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
50 50 99 99 8
```

세 방식의 차이를 한 문제에 몰아넣었다.

- `f1(&i)` : **주소**를 넘겨 `*p = 50` → 원본이 바뀐다 → **50**
- `f2(i)` : **값 복사**라 함수 안에서 60이 되어도 원본은 그대로 → **50**
- `f3(arr)` : **배열은 항상 포인터로 전달**된다 → `arr[0]` 이 **99**로 바뀐다
- `q` 는 `arr` 을 가리키므로 `*q` 도 같이 **99**, `*(q+3)` = `arr[3]` = **8**

`int p[]` 라고 써 있어도 실제로는 `int *p` 다. **배열 매개변수에는 값 복사가 없다.**

</details>

## Q4. 구조체 연결 리스트 순회

```c
#include <stdio.h>

typedef struct Node {
    int v;
    struct Node *next;
} Node;

int main(void) {
    Node c = {30, NULL};
    Node b = {20, &c};
    Node a = {10, &b};
    Node *p = &a;
    int s = 0;

    while (p != NULL) {
        s += p->v;
        p = p->next;
    }
    printf("%d ", s);

    p = &a;
    printf("%d %d", p->next->v, p->next->next->v);
    return 0;
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
60 20 30
```

`p = p->next` 로 사슬을 따라간다. 마지막 노드의 `next` 가 `NULL` 이라 반복이 끝난다.

- 합 : 10 + 20 + 30 = **60**
- `p->next->v` = b의 값 = **20**
- `p->next->next->v` = c의 값 = **30**

선언 순서가 c → b → a 인 이유는 **뒤 노드의 주소를 먼저 알아야** 앞 노드가 가리킬 수 있기 때문이다.

</details>

## Q5. 전위 · 중위 · 후위 순회 비교

```c
#include <stdio.h>

typedef struct T { int v; struct T *l, *r; } T;

void pre(T *n)  { if (!n) return; printf("%d", n->v); pre(n->l);  pre(n->r); }
void in(T *n)   { if (!n) return; in(n->l);  printf("%d", n->v); in(n->r); }
void post(T *n) { if (!n) return; post(n->l); post(n->r); printf("%d", n->v); }

int main(void) {
    T d = {4, NULL, NULL}, e = {5, NULL, NULL}, c = {3, NULL, NULL};
    T b = {2, &d, &e};
    T a = {1, &b, &c};
    pre(&a);  printf(" ");
    in(&a);   printf(" ");
    post(&a);
    return 0;
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
12453 42513 45231
```

같은 트리를 세 방식으로 순회한다.
```
      1
     / \
    2   3
   / \
  4   5
```
- **전위** (자기 → 왼 → 오) : 1 2 4 5 3 → `12453`
- **중위** (왼 → 자기 → 오) : 4 2 5 1 3 → `42513`
- **후위** (왼 → 오 → 자기) : 4 5 2 3 1 → `45231`

`printf` 가 재귀 호출들 사이 **어디에 있는지**만 보면 종류를 바로 알 수 있다.

</details>